# 2 つのシステムの差が有意かどうかを検定する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_evaluation/significance_test.ipynb)

NLP クックブック 16-10 ／ 評価指標と統計的検定

システム A が B より 1 ポイント高かった。この差は偶然なのか、本当に A のほうが良いと言えるのかを確かめたい。

In [1]:
import numpy as np

rng = np.random.default_rng(0)
n = 1000
gold = rng.integers(0, 2, n)
difficulty = rng.random(n)                                  # 事例ごとの難しさ (両システムに共通)
pred_a = np.where(difficulty + rng.normal(0, 0.15, n) < 0.86, gold, 1 - gold)
pred_b = np.where(difficulty + rng.normal(0, 0.15, n) < 0.83, gold, 1 - gold)
correct_a, correct_b = pred_a == gold, pred_b == gold
print(f"A = {correct_a.mean():.3f}  B = {correct_b.mean():.3f}  差 = {correct_a.mean() - correct_b.mean():+.3f}")

A = 0.858  B = 0.832  差 = +0.026


In [2]:
def paired_bootstrap(a, b, n_boot=10000):
    """対応のあるブートストラップ検定: 同じ事例を両システムについてリサンプリングし、A が B を上回らない割合を p 値とする"""
    idx = rng.integers(0, len(a), (n_boot, len(a)))
    diffs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    return (diffs <= 0).mean()

def approximate_randomization(a, b, n_perm=10000):
    """近似並べ替え検定: 事例ごとに A と B の結果をランダムに入れ替え、観測された差以上の差が出る割合を p 値とする (両側)"""
    observed = abs(a.mean() - b.mean())
    swap = rng.random((n_perm, len(a))) < 0.5
    diffs = np.abs(np.where(swap, b, a).mean(axis=1) - np.where(swap, a, b).mean(axis=1))
    return (diffs >= observed).mean()

print("対応のあるブートストラップ (片側) p =", paired_bootstrap(correct_a, correct_b))
print("近似並べ替え検定 (両側) p =", approximate_randomization(correct_a, correct_b))

対応のあるブートストラップ (片側) p = 0.0181
近似並べ替え検定 (両側) p = 0.0409


In [3]:
# 分類の正誤なら、McNemar 検定が使える (片方だけが正解した事例の数だけを見る)
from statsmodels.stats.contingency_tables import mcnemar

table = [[(correct_a & correct_b).sum(), (correct_a & ~correct_b).sum()],
         [(~correct_a & correct_b).sum(), (~correct_a & ~correct_b).sum()]]
print("A だけ正解:", table[0][1], " B だけ正解:", table[1][0], " McNemar p =", round(mcnemar(table, exact=True).pvalue, 4))

A だけ正解: 88  B だけ正解: 62  McNemar p = 0.0409


- 同じテストセットで 2 つのシステムを比べるときは、事例ごとの対応を使う「対応のある」検定にする。対応を無視すると検出力が大きく落ちる。
- p < 0.05 は「差がない場合にこの結果が出る確率が 5% 未満」という意味で、差の大きさや重要さは示さない。効果量と信頼区間も併せて報告する。
- たくさんのシステムや設定を比べると、偶然「有意」になるものが出てくる (多重比較)。Bonferroni 補正などを検討する。